# S&P 500 next-day — starter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/starters/sp500_starter.ipynb)

This is the starter for the **S&P 500 next-day** forecast. You submit a Python function
`predict(history)` that returns your forecast of **tomorrow's** index return; it's scored
**walk-forward on real prices**, folding in a new day after each US close. The sign of your forecast
is your directional bet; you're ranked by the Sharpe of those bets.

## Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q yfinance pandas numpy
import numpy as np, pandas as pd, yfinance as yf

## 1. Get the in-sample data (real ^GSPC closes)

In [ ]:
px = yf.download('^GSPC', start='2010-01-01', auto_adjust=True, progress=False)['Close']
if isinstance(px, pd.DataFrame):
    px = px.iloc[:, 0]
hist = px.to_frame('close')
print(hist.tail(3))
print('days:', len(hist))

## 2. Write your predictor

`predict(history)` receives a DataFrame of daily closes (column `close`) up to and including today,
and returns your forecast of **tomorrow's** return. This baseline is 5-day momentum — replace it.

In [ ]:
def predict(history):
    c = history['close']
    return float(c.iloc[-1] / c.iloc[-6] - 1)     # 5-day momentum (replace me)

## 3. Backtest it walk-forward (the same way it's scored)

In [ ]:
rets = hist['close'].pct_change().shift(-1)        # next-day realized return
preds = []
for i in range(200, len(hist) - 1):
    preds.append((hist.index[i], predict(hist.iloc[:i + 1]), rets.iloc[i]))
pdf = pd.DataFrame(preds, columns=['date', 'pred', 'next']).dropna()
bet = np.sign(pdf['pred'])                         # sign = direction; scored on the bet's return
pnl = bet * pdf['next']
print(f"days        : {len(pnl)}")
print(f"hit rate    : {(np.sign(pdf['pred'])==np.sign(pdf['next'])).mean():.1%}")
print(f"Sharpe (ann): {np.sqrt(252)*pnl.mean()/pnl.std():.2f}")

## 4. Submit

Paste your `predict(history)` into the competition's **Submit** form. It's re-scored daily on fresh,
genuinely out-of-sample data — no way to overfit the future.